# 🧪 Lab 5: 멀티 모델 Gateway 테스트

## 테스트 목표

APIM을 통해 **동일한 OpenAI 포맷**으로 Azure OpenAI와 Gemini를 호출하고,
응답이 정규화되는지 검증합니다.

| # | 테스트 | 기대 결과 |
|---|---|---|
| 1 | Azure OpenAI 호출 (기본) | 200 + OpenAI 응답 포맷 |
| 2 | Gemini 호출 (`x-model-provider: gemini`) | 200 + OpenAI 포맷으로 정규화된 응답 |
| 3 | 응답 포맷 비교 | 두 응답 모두 동일 구조 (`choices[0].message.content`) |
| 4 | 잘못된 Provider 헤더 | 기본 Azure OpenAI로 Fallback |

### 사전 준비

Lab 5 README의 1~4단계를 완료해야 합니다:
1. Gemini API Key를 Named Value로 저장
2. 모델 프로바이더별 라우팅 정책 적용
3. 요청/응답 형식 변환 정책 적용

> ⚠️ `.env`에 `GEMINI_API_KEY`가 설정되어 있어야 합니다.

In [ ]:
import os, json, time
import requests
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)

APIM_URL = os.getenv("APIM_URL")
APIM_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다."
assert APIM_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다."

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"

def call_gateway(prompt="Hello!", max_tokens=50, provider=None):
    """APIM Gateway를 통해 AI 모델 호출"""
    headers = {
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": APIM_KEY,
    }
    if provider:
        headers["x-model-provider"] = provider

    return requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers=headers,
        json={"messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens},
        timeout=30
    )

print("✅ 환경 설정 완료")
print(f"   APIM URL: {APIM_URL}")
print(f"   Base URL: {BASE_URL}")

---
## 테스트 1: Azure OpenAI 호출 (기본)

`x-model-provider` 헤더 없이 호출하면 기본 Azure OpenAI 백엔드로 라우팅됩니다.

In [ ]:
print("▶ 테스트 1: Azure OpenAI 호출 (기본)\n")

resp_openai = call_gateway(prompt="Say hello in Korean", max_tokens=50)

print(f"  HTTP Status: {resp_openai.status_code}")
if resp_openai.status_code == 200:
    data = resp_openai.json()
    content = data["choices"][0]["message"]["content"]
    model = data.get("model", "N/A")
    usage = data.get("usage", {})
    print(f"  모델: {model}")
    print(f"  응답: {content}")
    print(f"  토큰: prompt={usage.get('prompt_tokens', '?')}, completion={usage.get('completion_tokens', '?')}")
    print(f"\n  ✅ Azure OpenAI 정상 응답")
else:
    print(f"  ❌ 실패: {resp_openai.text[:200]}")

---
## 테스트 2: Gemini 호출 (`x-model-provider: gemini`)

**동일한 OpenAI 포맷**으로 요청하되, `x-model-provider: gemini` 헤더만 추가합니다.
APIM이 요청을 Gemini 포맷으로 변환하고, 응답을 다시 OpenAI 포맷으로 정규화합니다.

**핵심 검증:**
- 클라이언트 코드가 동일한데 백엔드만 Gemini로 변경
- 응답이 OpenAI 포맷(`choices[0].message.content`)으로 정규화

In [ ]:
print("▶ 테스트 2: Gemini 호출 (x-model-provider: gemini)\n")

resp_gemini = call_gateway(prompt="Say hello in Korean", max_tokens=50, provider="gemini")

print(f"  HTTP Status: {resp_gemini.status_code}")
if resp_gemini.status_code == 200:
    data = resp_gemini.json()
    content = data.get("choices", [{}])[0].get("message", {}).get("content", "N/A")
    model = data.get("model", "N/A")
    usage = data.get("usage", {})
    print(f"  모델: {model}")
    print(f"  응답: {content}")
    print(f"  토큰: prompt={usage.get('prompt_tokens', '?')}, completion={usage.get('completion_tokens', '?')}")
    print(f"\n  ✅ Gemini 응답이 OpenAI 포맷으로 정규화됨")
else:
    print(f"  ❌ 실패: {resp_gemini.text[:200]}")
    print(f"  → Gemini 정책 적용 여부를 확인하세요 (Lab 5 README 2~4단계)")

---
## 테스트 3: 응답 포맷 비교

Azure OpenAI와 Gemini의 응답 구조가 동일한지 비교합니다.

In [ ]:
print("▶ 테스트 3: 응답 포맷 비교\n")

def check_format(resp, name):
    """OpenAI 응답 표준 필드가 있는지 확인"""
    if resp.status_code != 200:
        return {"name": name, "valid": False, "reason": f"HTTP {resp.status_code}"}
    data = resp.json()
    fields = {
        "id": "id" in data,
        "object": data.get("object") == "chat.completion",
        "model": "model" in data,
        "choices": isinstance(data.get("choices"), list),
        "choices[0].message.content": bool(data.get("choices", [{}])[0].get("message", {}).get("content")),
        "usage.total_tokens": isinstance(data.get("usage", {}).get("total_tokens"), int),
    }
    return {"name": name, "valid": all(fields.values()), "fields": fields}

for result in [check_format(resp_openai, "Azure OpenAI"), check_format(resp_gemini, "Gemini")]:
    status = "✅" if result["valid"] else "❌"
    print(f"  {status} {result['name']}:")
    if "fields" in result:
        for field, ok in result["fields"].items():
            print(f"     {'✅' if ok else '❌'} {field}")
    else:
        print(f"     {result['reason']}")
    print()

if check_format(resp_openai, "")["valid"] and check_format(resp_gemini, "")["valid"]:
    print("✅ 두 응답 모두 동일한 OpenAI 포맷! 클라이언트 코드 변경 없이 백엔드 전환 가능.")
else:
    print("⚠️ 일부 필드가 누락되었습니다. 응답 변환 정책을 확인하세요.")

---
## 테스트 4: 잘못된 Provider 헤더 → Fallback

`x-model-provider: unknown` 처럼 정의되지 않은 값을 보내면
기본 Azure OpenAI 백엔드로 Fallback 되어야 합니다.

In [ ]:
print("▶ 테스트 4: 잘못된 Provider 헤더 → Fallback\n")

resp_fallback = call_gateway(prompt="Hi", max_tokens=10, provider="unknown")

print(f"  HTTP Status: {resp_fallback.status_code}")
if resp_fallback.status_code == 200:
    model = resp_fallback.json().get("model", "N/A")
    print(f"  모델: {model}")
    print(f"\n  ✅ 알 수 없는 Provider → Azure OpenAI로 정상 Fallback")
else:
    print(f"  ⚠️ {resp_fallback.status_code}: {resp_fallback.text[:100]}")

---
## 결과 요약

In [ ]:
print("═" * 55)
print("  멀티 모델 Gateway 테스트 요약")
print("═" * 55)
print()
print(f"  테스트 1 (Azure OpenAI):  {'✅ 정상' if resp_openai.status_code == 200 else '❌ 실패'}")
print(f"  테스트 2 (Gemini):        {'✅ 정상' if resp_gemini.status_code == 200 else '❌ 실패'}")

fmt_ok = check_format(resp_openai, '')["valid"] and check_format(resp_gemini, '')["valid"]
print(f"  테스트 3 (포맷 비교):     {'✅ 동일' if fmt_ok else '❌ 불일치'}")
print(f"  테스트 4 (Fallback):      {'✅ 정상' if resp_fallback.status_code == 200 else '❌ 실패'}")
print()
print("─" * 55)